# Delta Lake Optimization Techniques - Complete Guide

---

## 1. ANALYZE TABLE

### Purpose
Collects table-level and column-level statistics used by the **Spark query optimizer** (Catalyst) to generate optimal execution plans (e.g., choosing join strategies, estimating row counts).

### Syntax Variants
```sql
-- Table-level stats (row count + size)
ANALYZE TABLE catalog.schema.table COMPUTE STATISTICS;

-- Table-level stats without scanning data (size only)
ANALYZE TABLE catalog.schema.table COMPUTE STATISTICS NOSCAN;

-- Column-level stats (min, max, distinct_count, num_nulls, avg_len, max_len)
ANALYZE TABLE catalog.schema.table COMPUTE STATISTICS FOR COLUMNS col1, col2;
ANALYZE TABLE catalog.schema.table COMPUTE STATISTICS FOR ALL COLUMNS;

-- Delta-specific: Recompute file-level statistics (DBR 14.3+)
ANALYZE TABLE catalog.schema.table COMPUTE DELTA STATISTICS;

-- Storage metrics (DBR 18.0+)
ANALYZE TABLE catalog.schema.table COMPUTE STORAGE METRICS;
```

### What Statistics Are Collected
| Level | Statistics |
| --- | --- |
| Table-level | `sizeInBytes`, `numRows` |
| Column-level | `min`, `max`, `distinct_count`, `num_nulls`, `avg_col_len`, `max_col_len`, `histogram` (optional) |
| Delta Statistics | Per-file min/max, null counts, row counts for data skipping |
| Storage Metrics | `active_bytes`, `vacuumable_bytes`, `time_travel_bytes`, file counts |

### Where These Stats Are Stored
* **Table-level & column-level stats** → Stored in the **Hive Metastore / Unity Catalog metastore** (catalog properties)
* **Delta file-level stats** → Stored in the **Delta transaction log** (`_delta_log/` JSON and checkpoint Parquet files)

### When It's Useful
* Helps the optimizer choose between **Broadcast Join vs. Sort-Merge Join**
* Improves **cost-based optimization (CBO)** for complex query plans
* Required by **Predictive Optimization** for intelligent maintenance

---

## 2. OPTIMIZE (Bin-Packing / Compaction)

### Purpose
Rewrites small data files into larger, optimally-sized files (~1 GB default) to reduce I/O overhead and improve read performance.

### Syntax
```sql
-- Optimize entire table
OPTIMIZE catalog.schema.table;

-- Optimize specific partition
OPTIMIZE catalog.schema.table WHERE date = '2026-01-01';

-- Force full reclustering (DBR 16.4+, liquid clustering only)
OPTIMIZE catalog.schema.table FULL;
```

### Key Characteristics
* **Idempotent** - Running twice has no additional effect
* **Snapshot isolation** - Readers are not interrupted during OPTIMIZE
* **Incremental** - For liquid clustering, only rewrites data that needs reorganization
* **Returns stats** - Reports files removed/added and optimization metrics

### How It Works
1. Identifies small files (< target size)
2. Groups files by partition (if partitioned)
3. Rewrites into optimally-sized files
4. Atomically updates the transaction log
5. Old files become eligible for VACUUM after retention period

---

## 3. Z-ORDER

### Purpose
Co-locates related data in the same set of files using a multi-dimensional clustering technique (space-filling Z-curve). Maximizes **data skipping** effectiveness for multi-column filter predicates.

### Syntax
```sql
OPTIMIZE catalog.schema.table
ZORDER BY (col1, col2);
```

### How It Works
* Interleaves the byte ranges of multiple columns into a single ordering
* Data with similar values across Z-ordered columns ends up in the same files
* Files then have tight min/max ranges → more files skipped during queries

### Best Practices
* Z-order on **high-cardinality columns** used frequently in WHERE clauses
* Combine with partitioning: partition on **low-cardinality** columns, Z-order on **high-cardinality**
* Maximum ~4 columns recommended (diminishing returns beyond that)
* **Deprecated in favor of Liquid Clustering** (DBR 13.3+)

### Limitations
* Not incremental - rewrites all data each run
* Not compatible with liquid clustering
* Predictive optimization does NOT run Z-ORDER automatically

---

## 4. VACUUM

### Purpose
Removes data files that are **no longer referenced** by the current table version, reclaiming storage and reducing costs.

### Syntax
```sql
-- Default retention (7 days)
VACUUM catalog.schema.table;

-- Custom retention
VACUUM catalog.schema.table RETAIN 30 HOURS;

-- Dry run (preview files to delete)
VACUUM catalog.schema.table DRY RUN;
```

### Key Concepts
* **Retention period**: Controlled by `delta.deletedFileRetentionDuration` (default: 7 days)
* Files within the retention window are kept for **time travel** and **concurrent readers**
* After VACUUM, you cannot time-travel to versions older than the retention window
* **Never run concurrent VACUUMs** on the same table

### What Gets Deleted
* Files from previous versions (after UPDATE, DELETE, MERGE, OPTIMIZE)
* Files from failed/aborted transactions
* Does NOT delete `_delta_log` directory or checkpoint files

### Best Practice
```sql
-- Set retention before enabling predictive optimization
ALTER TABLE catalog.schema.table 
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '30 days');
```

---

## 5. Liquid Clustering

### Purpose
Replaces static partitioning and Z-ORDER with a **flexible, incremental** data layout strategy. Data is clustered by specified keys without rigid partition boundaries.

### Syntax
```sql
-- Enable during table creation
CREATE TABLE catalog.schema.table (
  id BIGINT, name STRING, region STRING, created_date DATE
) CLUSTER BY (region, created_date);

-- Enable on existing unpartitioned table
ALTER TABLE catalog.schema.table CLUSTER BY (region, created_date);

-- Automatic key selection (DBR 15.4+)
ALTER TABLE catalog.schema.table CLUSTER BY AUTO;

-- Change keys anytime (no full rewrite needed)
ALTER TABLE catalog.schema.table CLUSTER BY (new_col1, new_col2);

-- Disable clustering
ALTER TABLE catalog.schema.table CLUSTER BY NONE;

-- Trigger clustering
OPTIMIZE catalog.schema.table;

-- Force full reclustering (DBR 16.4+)
OPTIMIZE catalog.schema.table FULL;
```

### CLUSTER BY AUTO (Automatic Liquid Clustering)
* Databricks **intelligently selects** clustering keys based on your query workload patterns
* Requires DBR 15.4 LTS+ and Unity Catalog managed tables
* Keys are updated automatically as query patterns evolve
* Works with **Predictive Optimization** — no manual OPTIMIZE scheduling needed

### How Historical Data Gets Optimized
| Scenario | Behavior |
| --- | --- |
| Regular `OPTIMIZE` | **Incremental** — only rewrites data that needs clustering (new/unclustered data) |
| `OPTIMIZE FULL` (DBR 16.4+) | **Full reclustering** — rewrites ALL data to match current clustering keys |
| Key change (`ALTER TABLE ... CLUSTER BY`) | Subsequent OPTIMIZE only clusters new data; use `OPTIMIZE FULL` to recluster historical data |
| First-time enablement | Run `OPTIMIZE FULL` to cluster all existing data |

### Advantages Over Z-ORDER & Partitioning
* **Incremental** — only rewrites what's necessary
* **Flexible** — change keys without full rewrite
* **No partition skew** — handles uneven data distribution
* **Self-tuning** — with CLUSTER BY AUTO
* **Compatible with Predictive Optimization**

### Incompatibilities
* Cannot combine with partitioning
* Cannot combine with Z-ORDER
* Requires DBR 13.3 LTS+

---

## 6. File-Level Statistics (Delta Statistics)

### What They Are
Per-file metadata stored in the Delta transaction log that enables **data skipping** at query time.

### Statistics Collected Per File
| Statistic | Description |
| --- | --- |
| `minValues` | Minimum value for each column in the file |
| `maxValues` | Maximum value for each column in the file |
| `nullCount` | Number of null values per column |
| `numRecords` | Total number of records in the file |

### Where They Are Stored
* **Delta Transaction Log** (`_delta_log/` directory)
  * **JSON commit files** (e.g., `00000000000000000001.json`) — contain add/remove actions with stats
  * **Checkpoint Parquet files** (e.g., `00000000000000000010.checkpoint.parquet`) — consolidated stats for faster reads

### Configuration
```sql
-- Default: stats collected on first 32 columns (by schema order)
-- Change number of indexed columns
ALTER TABLE t SET TBLPROPERTIES('delta.dataSkippingNumIndexedCols' = '50');

-- Specify exact columns for stats (DBR 13.3+)
ALTER TABLE t SET TBLPROPERTIES('delta.dataSkippingStatsColumns' = 'col1, col2, col3');

-- Recompute stats after changing config (DBR 14.3+)
ANALYZE TABLE t COMPUTE DELTA STATISTICS;
```

### Important Notes
* For UC managed tables with **Predictive Optimization**, stats columns are chosen intelligently (no 32-column limit)
* Long strings are truncated during stats collection
* Timestamp/string columns have limited effectiveness in DBR < 16.4

---

## 7. Table-Level Statistics (Catalog Statistics)

### What They Are
Aggregate metadata about the entire table, used by the **Catalyst optimizer** for cost-based decisions.

### Statistics Available
| Statistic | Source |
| --- | --- |
| `sizeInBytes` | Computed from transaction log |
| `numRows` | From `ANALYZE TABLE ... COMPUTE STATISTICS` |
| Column `min`, `max` | From `ANALYZE TABLE ... FOR COLUMNS` |
| `distinct_count` | From `ANALYZE TABLE ... FOR COLUMNS` |
| `num_nulls` | From `ANALYZE TABLE ... FOR COLUMNS` |
| `histogram` | From `ANALYZE TABLE ... FOR COLUMNS` (advanced) |

### Where They Are Stored
* **Unity Catalog / Hive Metastore** metadata (catalog properties)
* Accessible via `DESCRIBE EXTENDED table_name`
* Viewable via `SHOW STATISTICS FROM table_name AS JSON` (DBR 17.0+)

---

## 8. Statistics Usage in Optimization Techniques

| Optimization Technique | Uses File-Level Stats | Uses Table-Level Stats |
| --- | --- | --- |
| Data Skipping / File Pruning | YES (min/max/nullCount) | No |
| Z-ORDER | YES (depends on stats for effectiveness) | No |
| Liquid Clustering | YES (requires Delta stats on clustering columns) | No |
| Partition Pruning | No (uses partition metadata) | No |
| Join Strategy Selection | No | YES (sizeInBytes, numRows) |
| Broadcast Join Threshold | No | YES (sizeInBytes) |
| Cost-Based Optimization | No | YES (all column stats) |
| Predicate Pushdown | No | No (Parquet-level metadata) |

---

## 9. When File Skipping (Data Skipping) Happens

### Mechanism
At query planning time, Delta Lake reads the file-level min/max stats from the transaction log and **eliminates files** whose value ranges don't overlap with the query predicates.

### Triggers (File Skipping Activates When)
* Query has **WHERE clause** filters on columns that have collected stats
* Columns are within the stats-collection scope (first 32 columns or configured list)
* Data is well-clustered (via OPTIMIZE, Z-ORDER, or Liquid Clustering)
* Stats exist (not null or missing)

### Operators That Benefit
```sql
-- Equality
WHERE col = 'value'

-- Range
WHERE col BETWEEN 100 AND 200

-- Comparison
WHERE col > '2026-01-01'

-- IN list
WHERE col IN ('A', 'B', 'C')

-- IS NULL / IS NOT NULL
WHERE col IS NOT NULL
```

### When File Skipping Does NOT Work
* Column is not in the stats collection scope
* Column has poor clustering (values spread across all files)
* Using functions on columns: `WHERE UPPER(col) = 'X'`
* Stats not collected (missing or disabled)
* String/timestamp columns on DBR < 16.4 (limited stats effectiveness)

---

## 10. Partitioning

### Purpose
Physically separates data into directories based on partition column values. Allows the engine to **skip entire directories** when queries filter on partition columns.

### Syntax
```sql
CREATE TABLE catalog.schema.table (
  id BIGINT, name STRING, event_date DATE, region STRING
) PARTITIONED BY (event_date);
```

### When To Use
* Very large tables (multi-TB)
* Clear low-cardinality filter column (e.g., date, region)
* High selectivity queries that always filter on partition column

### When NOT To Use
* High-cardinality columns → too many small files
* Frequently changing access patterns
* Tables < 1 TB (liquid clustering preferred)
* **Databricks recommends Liquid Clustering over partitioning for new tables**

---

## 11. Partition Pruning

### What It Is
The optimizer eliminates entire partitions (directories) from scanning based on WHERE clause predicates on partition columns.

### Static Partition Pruning
Filter values are known at **compile time** (plan optimization phase).

```sql
-- Static: literal value known at compile time
SELECT * FROM sales WHERE event_date = '2026-01-01';

-- Static: IN list
SELECT * FROM sales WHERE region IN ('US', 'EU');
```

**How it works:**
1. Optimizer sees the literal predicate
2. Directly maps to partition directories
3. Only matching partition directories are listed/scanned
4. Happens during **logical plan optimization**

### Dynamic Partition Pruning (DPP)
Filter values are determined at **runtime** from the result of a subquery or join.

```sql
-- Dynamic: partition filter values come from a subquery at runtime
SELECT * FROM fact_sales f
JOIN dim_date d ON f.date_key = d.date_key
WHERE d.year = 2026;
```

**How it works:**
1. The optimizer identifies that `date_key` is a partition column in `fact_sales`
2. At runtime, it first executes the filter on `dim_date` (small table)
3. Collects the resulting `date_key` values
4. **Broadcasts** these values as a runtime filter to prune partitions in `fact_sales`
5. Only matching partitions are scanned

### Static vs Dynamic Partition Pruning Comparison

| Aspect | Static Partition Pruning | Dynamic Partition Pruning |
| --- | --- | --- |
| Filter values known at | Compile time | Runtime |
| Triggered by | Literal predicates in WHERE | JOIN conditions or subqueries |
| Plan phase | Logical optimization | Physical execution |
| Requires broadcast | No | Yes (filter results broadcast) |
| Spark config | Always enabled | `spark.sql.optimizer.dynamicPartitionPruning.enabled` (default: true) |
| Works with | Partition columns only | Partition columns only |
| Overhead | None | Small (subquery execution + broadcast) |

### DPP Configurations
```sql
-- Enable/disable DPP (default: true)
SET spark.sql.optimizer.dynamicPartitionPruning.enabled = true;

-- Reuse broadcast results from joins (default: true)
SET spark.sql.optimizer.dynamicPartitionPruning.reuseBroadcastOnly = true;
```

---

## 12. Predictive Optimization

### Purpose
A fully managed, automated maintenance service for **Unity Catalog managed tables**. Eliminates manual scheduling of OPTIMIZE, VACUUM, and ANALYZE.

### What It Does Automatically
| Operation | Action |
| --- | --- |
| OPTIMIZE | Compacts small files + triggers liquid clustering |
| VACUUM | Removes unreferenced files (respects retention) |
| ANALYZE | Collects table/column statistics for CBO |
| Liquid Clustering Key Selection | Intelligently picks clustering keys (with CLUSTER BY AUTO) |

### Key Characteristics
* Enabled **by default** for accounts created after Nov 2024
* Uses **serverless compute** (billed as serverless jobs SKU)
* Works on Delta Lake and Iceberg managed tables
* Identifies tables that **benefit most** from maintenance
* Does NOT run Z-ORDER (ignores Z-ordered files)
* Stats columns chosen intelligently (no 32-column limit)

### Enabling
```sql
-- Enable at schema level
ALTER SCHEMA catalog.schema ENABLE PREDICTIVE OPTIMIZATION;

-- Enable at table level
ALTER TABLE catalog.schema.table ENABLE PREDICTIVE OPTIMIZATION;

-- Disable
ALTER TABLE catalog.schema.table DISABLE PREDICTIVE OPTIMIZATION;
```

### Best Practices
* Disable manual OPTIMIZE/VACUUM jobs when predictive optimization is active
* Set appropriate `delta.deletedFileRetentionDuration` before enabling
* Monitor via system tables: `system.storage.predictive_optimization_operations_history`

---

## 13. Additional Optimization Techniques

### Auto Compaction
* Runs **synchronously** after writes complete
* Combines small files within partitions (targets 128 MB)
* Only compacts files not previously compacted
* Independent of predictive optimization

```sql
-- Enable at table level
ALTER TABLE t SET TBLPROPERTIES('delta.autoOptimize.autoCompact' = 'auto');

-- Enable at session level
SET spark.databricks.delta.autoCompact.enabled = auto;
```

### Optimized Writes
* Automatically right-sizes files during write operations
* Reduces small files at write time (especially for partitioned tables)
* Adds a shuffle step before writing to coalesce small partitions

```sql
ALTER TABLE t SET TBLPROPERTIES('delta.autoOptimize.optimizeWrite' = 'true');
```

### Autotune File Size
* Databricks automatically determines target file size based on table size
* Small tables → smaller files (faster queries on small data)
* Large tables → larger files (reduce file listing overhead)

### Adaptive Query Execution (AQE)
Runtime optimizations that adapt the query plan based on actual data statistics:
* **Coalescing post-shuffle partitions** — reduces partition count when data is small
* **Splitting skewed partitions** — handles data skew in joins
* **Converting sort-merge join to broadcast** — runtime decision based on actual sizes
* **Converting sort-merge to shuffled hash join** — when beneficial

```sql
SET spark.sql.adaptive.enabled = true; -- enabled by default
```

### Predicate Pushdown / Filter Pushdown
* Pushes WHERE clause filters **down into the storage layer** (Parquet/ORC)
* Uses Parquet row-group level min/max stats for **row-group skipping**
* Different from Delta data skipping (which operates at the file level)
* Works on all columns, not just the first 32

### Bloom Filters (Delta Lake)
Probabilistic data structures for efficient equality lookups on high-cardinality columns.

```sql
CREATE BLOOMFILTER INDEX ON TABLE t FOR COLUMNS(col1 OPTIONS (fpp=0.1, numItems=1000000));
```

### Caching
* **Delta Cache** (SSD-based): Automatically caches hot Parquet files on local SSDs
* **Spark Cache** (`CACHE TABLE`): In-memory columnar caching for repeatedly accessed data

---

## 14. Complete Statistics Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                    QUERY EXECUTION                               │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌───────────────┐     ┌──────────────────┐    ┌────────────┐  │
│  │ Partition     │     │ File Skipping    │    │ Row-Group  │  │
│  │ Pruning       │     │ (Data Skipping)  │    │ Skipping   │  │
│  │               │     │                  │    │            │  │
│  │ Uses:         │     │ Uses:            │    │ Uses:      │  │
│  │ Partition     │     │ Delta Log Stats  │    │ Parquet    │  │
│  │ Metadata      │     │ (min/max/null)   │    │ Footer     │  │
│  └───────┬───────┘     └────────┬─────────┘    └─────┬──────┘  │
│          │                      │                     │         │
│     Directory              File Level            Row-Group      │
│     Elimination            Elimination           Elimination    │
│                                                                 │
├─────────────────────────────────────────────────────────────────┤
│                    PLAN OPTIMIZATION (CBO)                       │
│                                                                 │
│  Uses: Table-level stats from ANALYZE TABLE                     │
│  Stored in: Unity Catalog / Hive Metastore                      │
│  Purpose: Join strategy, broadcast decisions, row estimates     │
└─────────────────────────────────────────────────────────────────┘
```

---

## 15. Summary Decision Matrix

| Technique | When to Use | Target File Size | Incremental? | Auto via Predictive Opt? |
| --- | --- | --- | --- | --- |
| OPTIMIZE (bin-pack) | Many small files | ~1 GB | No (rewrites all small files) | YES |
| Z-ORDER | Multi-column filters, legacy tables | ~1 GB | No | NO |
| Liquid Clustering | New tables, flexible filtering | Auto-tuned | YES | YES |
| VACUUM | Reduce storage costs | N/A | N/A | YES |
| ANALYZE | Complex joins, CBO needed | N/A | N/A | YES |
| Partitioning | Very large tables, low-cardinality filter | N/A | N/A | No |
| Auto Compaction | Streaming/frequent writes | 128 MB | Yes | Independent |
| Optimized Writes | Partitioned tables with many small writes | Auto | Yes | N/A |
| Bloom Filters | High-cardinality equality lookups | N/A | N/A | No |